# FT-03 : Supervised Fine-Tuning (SFT) — Instruction Following

**Objectif** : Apprendre le Supervised Fine-Tuning (SFT), la technique qui transforme un modele de base (base model) en un assistant capable de suivre des instructions (instruct model). Vous comprendrez le format instruction-réponse, la preparation des donnees d'entrainement, et l'evaluation de la qualite des reponses.

**Prerequis** :
- FT-01 (Introduction au Fine-Tuning, LoRA sur GPT-2)
- FT-02 (QLoRA Quantization)
- Notions de PyTorch et Transformers

**Plan du notebook** :
1. Base Model vs Instruct Model — la difference cruciale
2. Le format instruction-réponse (chat templates)
3. Preparation d'un dataset SFT
4. SFT avec QLoRA sur OPT-1.3B
5. Evaluation qualitative : avant vs apres
6. Impact de la taille du dataset
7. Exercices

**Duree estimee** : 45-60 minutes

In [1]:
import torch
import os
import gc
import json
import time

print(f"PyTorch {torch.__version__}")
print(f"CUDA : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM : {props.total_memory / 1e9:.1f} Go")

# Mode batch pour execution automatisee
BATCH_MODE = os.environ.get("BATCH_MODE", "false").lower() == "true"
print(f"Batch mode : {BATCH_MODE}")

PyTorch 2.11.0+cu126
CUDA : True
GPU : NVIDIA GeForce RTX 3090
VRAM : 25.8 Go
Batch mode : False


## 1. Base Model vs Instruct Model

Un **modele de base** (base model) est entraine uniquement a predire le token suivant dans un texte. Il ne comprend pas la notion d'"instruction" — si vous lui dites "Expliquez la photosynthese", il pourrait continuer la phrase ("...en utilisant des termes simples") au lieu de repondre.

Un **modele instruct** (instruct model) a subi un Supervised Fine-Tuning sur des paires instruction-réponse. Il a appris a :
- Comprendre qu'on lui pose une question ou donne une tache
- Produire une reponse structuree et pertinente
- Suivre des formats specifiques (explication, code, liste, etc.)

La transition base → instruct est exactement ce que nous allons reproduire dans ce notebook avec un petit modele.

## 2. Le format instruction-réponse (chat templates)

Les modeles instruct utilisent un **format de conversation** (chat template) pour delimiter les roles. Par exemple :

```
### Human: Expliquez la photosynthese en 2 phrases.
### Assistant: La photosynthese est le processus par lequel les plantes convertissent la lumiere du soleil, le CO2 et l'eau en glucose et oxygene. Ce mecanisme se deroule dans les chloroplastes des cellules vegetales grace a la chlorophylle.
```

Le format exact depend du modele. Pour les modeles de la famille OPT, le format `### Human:` / `### Assistant:` est courant. D'autres modeles utilisent des tokens speciaux (`<|im_start|>user\n...<|im_end|>` pour les modeles ChatML).

**Point cle** : le chat template est un contrat entre les donnees d'entrainement et l'inference. Si vous entrainez avec un format, vous devez utiliser le meme format a l'inference.

In [2]:
# Charger le modele de base (OPT-1.3B, quantize 4-bit pour economiser la VRAM)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "facebook/opt-1.3b"

# Quantization 4-bit (QLoRA, comme dans FT-02)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Chargement de {MODEL_NAME} en 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
print(f"Modele charge. Parametres : {sum(p.numel() for p in model_base.parameters()):,}")

if torch.cuda.is_available():
    vram_mb = torch.cuda.memory_allocated() / 1e6
    print(f"VRAM utilisee : {vram_mb:.0f} Mo")

Chargement de facebook/opt-1.3b en 4-bit...


W0524 10:30:11.770000 47108 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Modele charge. Parametres : 711,778,304
VRAM utilisee : 371 Mo


### 2a. Demonstration : le modele de base ne suit pas les instructions

Testons le modele de base avec des instructions. Observez comment il "continue" le texte au lieu de repondre a la question.

In [3]:
# Test du modele de base avec des instructions
def generate(model, prompt, max_new_tokens=80, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

# Instructions de test
test_prompts = [
    "### Human: Qu'est-ce que la gravite ?\n### Assistant:",
    "### Human: Expliquez le concept d'intelligence artificielle en termes simples.\n### Assistant:",
    "### Human: Donne-moi 3 conseils pour ameliorer ma productivite.\n### Assistant:",
]

print("=== REPONSES DU MODELE DE BASE (avant SFT) ===\n")
for prompt in test_prompts:
    response = generate(model_base, prompt)
    instruction = prompt.split("### Human: ")[1].split("\n")[0]
    print(f"Instruction : {instruction}")
    print(f"Reponse     : {response}")
    print("-" * 60)

=== REPONSES DU MODELE DE BASE (avant SFT) ===



C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Instruction : Qu'est-ce que la gravite ?
Reponse     : Qu'est-ce que la gravite ?
### Assistant: Qu'est-ce que la gravite ?
### Assistant: Qu'est-ce que la gravite ?
### Assistant: Qu'est-ce que la gravite ?
------------------------------------------------------------


Instruction : Expliquez le concept d'intelligence artificielle en termes simples.
Reponse     : Pour le détail, expliquez la capacité de reconnaître les différences entre des types d'intelligence artificielle.
------------------------------------------------------------


Instruction : Donne-moi 3 conseils pour ameliorer ma productivite.
Reponse     : Si j'ai vécu un meilleur jour, j'ai vécu un meilleur mois.
### Assistant: Si j'ai vécu un meilleur jour, j'ai vécu un meilleur mois.   ## Assistant: Si j'ai vécu un meilleur jour, j'ai vécu
------------------------------------------------------------


**Observation** : Le modele de base produit du texte coherent mais ne repond pas vraiment aux instructions. Il peut ignorer la question, la reformuler, ou generer du contenu sans rapport. C'est normal : il a ete entraine uniquement a predire la suite du texte, pas a suivre des instructions.

Nous allons maintenant lui apprendre a repondre correctement avec le SFT.

## 3. Preparation d'un dataset SFT

Pour le SFT, nous avons besoin de paires **instruction → reponse attendue**. La qualite du dataset est le facteur le plus important pour la reussite du fine-tuning.

Nous allons creer un petit dataset synthetique en francais couvrant plusieurs categories d'instructions :
- **Questions factuelles** : "Qu'est-ce que X ?"
- **Explications** : "Expliquez le concept de Y"
- **Listes/conseils** : "Donnez N conseils pour Z"
- **Resumes** : "Resumez le texte suivant"

In [4]:
# Dataset SFT synthetique — paires instruction/reponse en francais
# Format: {"instruction": "...", "response": "..."}
sft_data = [
    # Questions factuelles
    {"instruction": "Qu'est-ce que la gravite ?", "response": "La gravite est une force fondamentale qui attire deux corps massifs l'un vers l'autre. Sur Terre, elle nous maintient au sol et donne leur poids aux objets. Isaac Newton a formule la loi de la gravitation universelle, puis Albert Einstein l'a reformulee dans sa theorie de la relativite generale comme une deformation de l'espace-temps."},
    {"instruction": "Qu'est-ce que l'intelligence artificielle ?", "response": "L'intelligence artificielle (IA) est un domaine de l'informatique qui vise a creer des systemes capables de realiser des taches necessitant normalement l'intelligence humaine : reconnaissance d'images, comprehension du langage, prise de decision. L'apprentissage profond (deep learning) est la technique la plus performante actuellement."},
    {"instruction": "Qu'est-ce que le machine learning ?", "response": "Le machine learning (apprentissage automatique) est une branche de l'IA ou les systemes apprennent a partir de donnees sans etre explicitement programmes. Les trois types principaux sont : l'apprentissage supervise (avec labels), non supervise (sans labels) et par renforcement (par recompenses)."},
    {"instruction": "Qu'est-ce qu'un neurone artificiel ?", "response": "Un neurone artificiel est l'unite de base d'un reseau de neurones. Il recoit des entrees numeriques, les multiplie par des poids, additionne les resultats, puis applique une fonction d'activation (comme ReLU ou sigmoid) pour produire une sortie. En combinant des millions de neurones, on obtient des modeles capables d'apprendre des patterns complexes."},
    {"instruction": "Qu'est-ce que Python ?", "response": "Python est un langage de programmation de haut niveau, cree par Guido van Rossum en 1991. Il est connu pour sa syntaxe lisible et sa polyvalence. Python est tres utilise en science des donnees, en intelligence artificielle, en developpement web et en automatisation."},

    # Explications
    {"instruction": "Expliquez le concept de reseau de neurones en termes simples.", "response": "Un reseau de neurones fonctionne comme une equipe de decision. Imaginez des centaines de personnes, chacune regardant une petite partie d'un probleme. Chaque personne donne son avis avec une certaine confiance (le poids). Un chef d'equipe rassemble tous les avis et prend une decision. En s'entrainant sur beaucoup d'exemples, l'equipe apprend a donner les bonnes reponses."},
    {"instruction": "Expliquez la difference entre CPU et GPU.", "response": "Le CPU (processeur) est comme un chef tres intelligent qui traite les taches une par une rapidement. Le GPU (carte graphique) est comme une equipe de milliers de travailleurs moins rapides mais travaillant en parallele. Pour les calculs repetitifs comme l'entrainement des reseaux de neurones, le GPU est beaucoup plus efficace grace a son parallelisme massif."},
    {"instruction": "Expliquez ce qu'est une API.", "response": "Une API (Application Programming Interface) est comme un menu de restaurant entre deux logiciels. Le menu liste ce que vous pouvez commander (les fonctions disponibles), comment passer commande (les parametres), et ce que vous recevez (la reponse). Vous n'avez pas besoin de savoir comment la cuisine fonctionne, il suffit de commander."},

    # Listes / Conseils
    {"instruction": "Donnez 3 conseils pour ameliorer sa productivite au travail.", "response": "Voici 3 conseils pour ameliorer votre productivite :\n1. **Technique Pomodoro** : Travaillez 25 minutes, puis faites 5 minutes de pause. Apres 4 cycles, prenez une pause de 15-30 minutes.\n2. **Priorisation** : Commencez par la tache la plus importante (la methode \"eat the frog\"). Les premieres heures sont souvent les plus productives.\n3. **Bloquer les distractions** : Desactivez les notifications, fermez les onglets inutiles et utilisez un casque pour signaler que vous etes concentre."},
    {"instruction": "Listez 4 avantages du langage Python pour la data science.", "response": "Voici 4 avantages de Python pour la data science :\n1. **Ecosysteme riche** : NumPy, Pandas, Scikit-learn, TensorFlow, PyTorch couvrent tout le pipeline.\n2. **Syntaxe lisible** : Le code est facile a comprendre et a partager entre equipes.\n3. **Communaute active** : Des millions de developpeurs, des milliers de tutoriels et de packages.\n4. **Polyvalence** : Meme langage pour l'analyse, le ML, le web et l'automatisation."},
    {"instruction": "Quelles sont les 5 etapes d'un projet de machine learning ?", "response": "Les 5 etapes d'un projet de ML sont :\n1. **Definition du probleme** : Clarifier l'objectif et les metriques de succes.\n2. **Collecte de donnees** : Obtenir un dataset representative et de qualite.\n3. **Preparation** : Nettoyage, transformation, et split train/validation/test.\n4. **Entrainement** : Choisir un modele, l'entrainer et optimiser les hyperparametres.\n5. **Evaluation et deploiement** : Mesurer les performances et mettre en production."},

    # Resumes
    {"instruction": "Resumez en une phrase : Le machine learning permet aux ordinateurs d'apprendre a partir de donnees sans etre explicitement programmes pour chaque tache.", "response": "Le machine learning est une approche ou les systemes informatiques apprennent automatiquement des patterns a partir de donnees plutot que de suivre des regles predefinies."},
    {"instruction": "Resumez en une phrase : Les transformers sont une architecture de reseau de neurones qui utilise le mecanisme d'attention pour traiter des sequences de donnees en parallele, revolutionnant le NLP.", "response": "Les transformers utilisent le mecanisme d'attention pour traiter des sequences en parallele, ce qui a transforme le traitement du langage naturel."},
    {"instruction": "Resumez en une phrase : Le fine-tuning consiste a ajuster les poids d'un modele pre-entraine sur un dataset specifique pour adapter ses performances a une tache particuliere.", "response": "Le fine-tuning adapte un modele pre-entraine a une tache specifique en ajustant ses poids sur un dataset cible."},

    # Questions conceptuelles
    {"instruction": "Pourquoi le deep learning a-t-il connu un essor recent ?", "response": "Le deep learning a connu un essor recent grace a trois facteurs : (1) la disponibilite de grandes quantites de donnees sur Internet, (2) l'augmentation de la puissance de calcul, notamment avec les GPU, et (3) les progres algorithmiques comme les architectures transformers et les techniques d'entrainement plus efficaces."},
    {"instruction": "Quelle est la difference entre apprentissage supervise et non supervise ?", "response": "En apprentissage supervise, le modele apprend a partir de donnees labellisees (exemples avec la reponse correcte), comme classifier des images avec leurs etiquettes. En apprentissage non supervise, le modele decouvre des structures dans les donnees sans labels, comme regrouper des clients similaires (clustering)."},
    {"instruction": "Qu'est-ce que le surapprentissage (overfitting) ?", "response": "Le surapprentissage se produit quand un modele apprend par coeur les donnees d'entrainement au lieu de generaliser. Il obtient d'excellents resultats sur les donnees vues mais de mauvais sur les nouvelles donnees. On le combat avec la regularisation, le dropout, l'augmentation de donnees et un split train/validation."},

    # Code / Pratique
    {"instruction": "Comment creer une liste vide en Python ?", "response": "Pour creer une liste vide en Python, vous pouvez utiliser soit les crochets `ma_liste = []` soit le constructeur `ma_liste = list()`. La premiere methode est plus courante et legerement plus rapide."},
    {"instruction": "Quelle est la difference entre une liste et un tuple en Python ?", "response": "Une liste (entre crochets `[]`) est mutable : on peut ajouter, supprimer ou modifier ses elements. Un tuple (entre parentheses `()`) est immutable : une fois cree, il ne peut pas etre modifie. Les tuples sont legerement plus rapides et servent souvent de cles de dictionnaire."},
    {"instruction": "Qu'est-ce qu'un dictionnaire en Python ?", "response": "Un dictionnaire est une structure de donnees qui stocke des paires cle-valeur. Par exemple : `etudiant = {\"nom\": \"Alice\", \"age\": 22}`. On accede aux valeurs via les cles : `etudiant[\"nom\"]` retourne \"Alice\". Les dictionnaires sont mutables et tres efficaces pour la recherche."},
]

print(f"Dataset SFT cree : {len(sft_data)} paires instruction/reponse")
print(f"\nExemple :")
print(f"  Instruction : {sft_data[0]['instruction']}")
print(f"  Reponse     : {sft_data[0]['response'][:80]}...")

# Statistiques
categories = {
    "Questions factuelles": sum(1 for d in sft_data if d["instruction"].startswith("Qu'est-ce")),
    "Explications": sum(1 for d in sft_data if d["instruction"].startswith("Expliquez")),
    "Listes/Conseils": sum(1 for d in sft_data if any(k in d["instruction"] for k in ["conseils", "Listez", "etapes", "avantages"])),
    "Resumes": sum(1 for d in sft_data if d["instruction"].startswith("Resumez")),
    "Concepts": sum(1 for d in sft_data if d["instruction"].startswith("Pourquoi") or d["instruction"].startswith("Quelle")),
    "Code/Pratique": sum(1 for d in sft_data if any(k in d["instruction"] for k in ["Python", "Comment", "creer"])),
}
print(f"\nRepartition par categorie :")
for cat, count in categories.items():
    print(f"  {cat} : {count}")

Dataset SFT cree : 20 paires instruction/reponse

Exemple :
  Instruction : Qu'est-ce que la gravite ?
  Reponse     : La gravite est une force fondamentale qui attire deux corps massifs l'un vers l'...

Repartition par categorie :
  Questions factuelles : 7
  Explications : 3
  Listes/Conseils : 3
  Resumes : 3
  Concepts : 4
  Code/Pratique : 5


## 4. Formatage et tokenisation des donnees

Avant l'entrainement, chaque paire instruction/reponse doit etre formatee selon le **chat template** du modele. Pour OPT, le format est :

```
### Human: {instruction}
### Assistant: {response}
```

Nous tokenisons ensuite le texte formate pour obtenir les sequences d'entiers compréhensibles par le modele.

In [5]:
# Formater les donnees SFT en utilisant le chat template
def format_sft_example(example):
    """Convertit une paire instruction/reponse en texte formate pour OPT."""
    return f"### Human: {example['instruction']}\n### Assistant: {example['response']}"

# Formater et tokeniser toutes les donnees
formatted_data = []
for item in sft_data:
    text = format_sft_example(item)
    formatted_data.append(text)

print(f"Exemple formate :\n{'-' * 50}")
print(formatted_data[0][:200] + "...")
print(f"\n{len(formatted_data)} exemples formates")

# Tokeniser avec le tokenizer du modele
tokenized_data = []
for text in formatted_data:
    tokens = tokenizer(text, truncation=True, max_length=512, return_tensors=None)
    tokenized_data.append({
        "input_ids": tokens["input_ids"],
        "attention_mask": tokens["attention_mask"],
        "text": text,
    })

# Statistiques de longueur
lengths = [len(t["input_ids"]) for t in tokenized_data]
print(f"\nStatistiques de tokenisation :")
print(f"  Longueur min  : {min(lengths)} tokens")
print(f"  Longueur max  : {max(lengths)} tokens")
print(f"  Longueur moy  : {sum(lengths) / len(lengths):.0f} tokens")

Exemple formate :
--------------------------------------------------
### Human: Qu'est-ce que la gravite ?
### Assistant: La gravite est une force fondamentale qui attire deux corps massifs l'un vers l'autre. Sur Terre, elle nous maintient au sol et donne leur poids au...

20 exemples formates

Statistiques de tokenisation :
  Longueur min  : 80 tokens
  Longueur max  : 174 tokens
  Longueur moy  : 118 tokens


### 4a. Configuration QLoRA pour le SFT

Nous reprenons la configuration QLoRA du FT-02, en ajoutant les adapteurs LoRA sur le modele deja quantifie en 4-bit. Seuls les parametres LoRA seront entraines (quelques millions), pas le modele complet (~1.3 milliard).

In [6]:
# Configurer QLoRA (LoRA sur modele quantifie)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Preparer le modele pour l'entrainement en 4-bit
model_base = prepare_model_for_kbit_training(model_base)

# Configuration LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                    # Rang des matrices LoRA
    lora_alpha=32,           # Facteur de mise a l'echelle
    lora_dropout=0.05,       # Dropout pour regularisation
    target_modules=["q_proj", "v_proj"],  # Couches a adapter
    bias="none",
)

# Appliquer LoRA au modele
model_sft = get_peft_model(model_base, lora_config)
model_sft.print_trainable_parameters()

print(f"\nVRAM avant entrainement : {torch.cuda.memory_allocated() / 1e6:.0f} Mo")

trainable params: 3,145,728 || all params: 1,318,903,808 || trainable%: 0.2385

VRAM avant entrainement : 598 Mo


## 5. Entrainement SFT

Nous utilisons le `Trainer` de HuggingFace pour orchestrer l'entrainement. Le dataset est petit (21 exemples), donc nous ferons plusieurs epoques pour que le modele apprenne bien le format instruction/reponse.

**Hyperparametres cles** :
- `num_train_epochs=5` : le dataset est petit, on compense avec plus de passages
- `learning_rate=2e-4` : taux d'apprentissage standard pour LoRA
- `per_device_train_batch_size=2` : petit batch pour la VRAM
- `gradient_accumulation_steps=4` : simule un batch de 8

In [7]:
# Preparation du dataset pour le Trainer HuggingFace
from datasets import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Creer un Dataset HuggingFace a partir des donnees formatees
train_texts = [t["text"] for t in tokenized_data]
train_dataset = Dataset.from_dict({"text": train_texts})

# Fonction de tokenisation pour le dataset
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
        return_tensors=None,
    )

tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

# Data collator pour le language modeling causal
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

print(f"Dataset pret : {len(tokenized_dataset)} exemples")
print(f"Colonnes : {tokenized_dataset.column_names}")

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Dataset pret : 20 exemples
Colonnes : ['input_ids', 'attention_mask']


In [8]:
# Entrainement SFT
training_args = TrainingArguments(
    output_dir="./results_ft03",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=5,
    save_strategy="epoch",
    report_to="none",
    fp16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model_sft,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Debut de l'entrainement SFT...")
start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time

print(f"\nEntrainement termine en {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"Perte finale : {train_result.training_loss:.4f}")
print(f"VRAM apres entrainement : {torch.cuda.memory_allocated() / 1e6:.0f} Mo")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Debut de l'entrainement SFT...


Step,Training Loss
5,3.209294
10,2.853980
15,2.708146



Entrainement termine en 31.4s (0.5 min)
Perte finale : 2.9238
VRAM apres entrainement : 614 Mo


## 6. Evaluation : avant vs apres SFT

C'est le moment de verifier si le SFT a fonctionne. Nous testons le modele fine-tune avec les memes instructions que le modele de base. Un modele SFT reussi devrait :
- Repondre directement a l'instruction (pas continuer le texte)
- Produire une reponse coherente et pertinente
- Respecter le format attendu (explication, liste, resume)

In [9]:
# Evaluation comparative : modele de base vs modele SFT
model_sft.eval()

# Instructions de test (memes que pour le modele de base, plus de nouvelles)
eval_prompts = [
    "### Human: Qu'est-ce que la gravite ?\n### Assistant:",
    "### Human: Expliquez le concept d'intelligence artificielle en termes simples.\n### Assistant:",
    "### Human: Donne-moi 3 conseils pour ameliorer ma productivite.\n### Assistant:",
    "### Human: Qu'est-ce qu'un reseau de neurones ?\n### Assistant:",
]

print("=" * 70)
print("COMPARAISON : MODELE DE BASE vs MODELE SFT")
print("=" * 70)

for prompt in eval_prompts:
    instruction = prompt.split("### Human: ")[1].split("\n")[0]

    # Reponse du modele SFT
    response_sft = generate(model_sft, prompt, max_new_tokens=100, temperature=0.7)

    print(f"\nInstruction : {instruction}")
    print(f"  SFT : {response_sft[:200]}")
    print("  " + "-" * 60)

COMPARAISON : MODELE DE BASE vs MODELE SFT


C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Instruction : Qu'est-ce que la gravite ?
  SFT : A gravite, un objectif est établi par les images de l'objet. Ces images sont les plus longues pour l'apprentissage de la forme, mais elles sont très peu utilisées.
### Human: Parce que les images sont
  ------------------------------------------------------------



Instruction : Expliquez le concept d'intelligence artificielle en termes simples.
  SFT : Le concept d'intelligence artificielle est un système de machine learning. Les robots ne sont pas nécessaires à l'intelligence artificielle. Les robots ne peuvent plus l'apprendre et leurs actions ne 
  ------------------------------------------------------------



Instruction : Donne-moi 3 conseils pour ameliorer ma productivite.
  SFT : Ne pas pousser la petite quantitatrice.
### Human: Ne pas pousser la petite quantitatrice.
### Assistant: Ne pas pousser la petite quantitatrice.
### Human: Ne pas pousser la petite quantitatrice.
###
  ------------------------------------------------------------



Instruction : Qu'est-ce qu'un reseau de neurones ?
  SFT : Un reseau de neurones, c'est une association de neurones, et c'est un ensemble de neurones qui vient de la même source.
### Human: A quoi ressemblerait un reseau de neurones ?
### Assistant: Un reseau
  ------------------------------------------------------------


**Observation** : Le modele SFT devrait maintenant repondre directement aux instructions au lieu de continuer le texte. La qualite des reponses depend de :
1. **La taille du dataset** — 21 exemples est un minimum, les modeles de production utilisent 10K-100K+ exemples
2. **La qualite des donnees** — des reponses mal formatees ou incorrectes seront apprises telles quelles
3. **Le nombre d'epochs** — trop peu = sous-apprentissage, trop = surapprentissage

## 7. Impact de la taille du dataset

Un SFT efficace depends fortement de la quantite de donnees. Testons avec differents sous-ensembles de notre dataset pour observer l'impact sur la qualite des reponses.

In [10]:
# Impact de la taille du dataset — entrainement avec seulement 5 exemples
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import copy

# Recharger un modele frais pour comparaison equitable
model_small = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model_small = prepare_model_for_kbit_training(model_small)
model_small = get_peft_model(model_small, lora_config)

# Entrainer avec seulement 5 exemples
small_texts = [format_sft_example(d) for d in sft_data[:5]]
small_dataset = Dataset.from_dict({"text": small_texts})
small_tokenized = small_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

small_trainer = Trainer(
    model=model_small,
    args=TrainingArguments(
        output_dir="./results_ft03_small",
        num_train_epochs=5,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        logging_steps=2,
        report_to="none",
        fp16=True,
        gradient_checkpointing=True,
        remove_unused_columns=False,
    ),
    train_dataset=small_tokenized,
    data_collator=data_collator,
)

print("Entrainement avec 5 exemples seulement...")
small_result = small_trainer.train()
print(f"Perte finale (5 exemples) : {small_result.training_loss:.4f}")

# Tester
test_prompt = "### Human: Qu'est-ce que la gravite ?\n### Assistant:"
response_small = generate(model_small, test_prompt, max_new_tokens=80)
print(f"\nReponse (5 exemples) : {response_small[:200]}")

# Liberer la memoire
del model_small, small_trainer
gc.collect()
torch.cuda.empty_cache()
print(f"\nVRAM apres cleanup : {torch.cuda.memory_allocated() / 1e6:.0f} Mo")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Entrainement avec 5 exemples seulement...


Step,Training Loss
2,3.081617
4,2.941506
6,2.768976
8,2.600261
10,2.501075


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


[transformers] Caching is incompatible with gradient checkpointing in OPTDecoderLayer. Setting `past_key_values=None`.


Perte finale (5 exemples) : 2.7787


C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Reponse (5 exemples) : La homhomig-----(,,,,,s. (e. ---/ in,"--im,,I(
. ha e orEAEAEAE AEMBB..‣ I C  () D fast on,,,, and firm firm computer and and in in in,,; Ge Ge Ge Ge Ge

VRAM apres cleanup : 614 Mo


**Conclusion sur la taille du dataset** : Avec seulement 5 exemples, le modele apprend moins bien le format instruction/reponse. La difference est visible : les reponses sont moins structurees et le modele peut encore parfois \"oublier\" qu'il doit repondre a une instruction.

En production, les datasets SFT contiennent typiquement entre 10 000 et 1 million de paires instruction/reponse.

In [11]:
# Liberation de la memoire GPU
del model_sft, model_base, trainer
gc.collect()
torch.cuda.empty_cache()
print("Memoire GPU liberee.")
if torch.cuda.is_available():
    print(f"VRAM residuelle : {torch.cuda.memory_allocated() / 1e6:.0f} Mo")

Memoire GPU liberee.
VRAM residuelle : 19 Mo


## 8. Exercices

Mettez en pratique les concepts de ce notebook. Chaque exercice build sur les concepts precedents.

### Exercice 1 : Creer un dataset SFT thematique

Creez un dataset SFT de 10 paires instruction/reponse sur un **theme de votre choix** (cuisine, sport, musique, etc.). Formatez les donnees avec le chat template, puis entrainez un modele QLoRA et observez les resultats.

**Etapes** :
1. Choisissez un theme et creez 10 paires instruction/reponse coherentes
2. Formatez avec le chat template `### Human:` / `### Assistant:`
3. Entrainez le modele avec les memes hyperparametres
4. Testez avec 3 instructions de votre theme

In [12]:
# Exercice 1 : Creez votre dataset SFT thematique
# TODO etudiant : remplacez les exemples ci-dessous par votre propre theme

mon_dataset = [
    {"instruction": "Donnez une recette simple de pates.", "response": "Faites cuire 200g de pates dans l'eau bouillante salee pendant 8-10 minutes. Egouttez, ajoutez un filet d'huile d'olive, du parmesan rape et du poivre. C'est pret !"},
    # TODO etudiant : ajoutez 9 autres paires sur votre theme
]

# TODO etudiant : formatez avec le chat template et entrainez le modele
print(f"Dataset en cours : {len(mon_dataset)} paires (objectif : 10)")

Dataset en cours : 1 paires (objectif : 10)


### Exercice 2 : Comparer differents rangs LoRA

Le rang `r` des matrices LoRA controle la capacite d'apprentissage. Testez avec `r=4` (faible) et `r=64` (eleve) et comparez les resultats. Observez le trade-off entre qualite des reponses et nombre de parametres entrainables.

**Etapes** :
1. Entrainez avec `r=4` et notez le nombre de parametres entrainables + la perte finale
2. Entrainez avec `r=64` et notez les memes metriques
3. Comparez les reponses sur 2 instructions test
4. Quel rang donne les meilleurs resultats sur notre petit dataset ?

In [13]:
# Exercice 2 : Comparez differents rangs LoRA (r=4 vs r=64)
# TODO etudiant : entrainez avec r=4 puis r=64 et comparez

# Indice : modifiez LoraConfig(r=4, ...) et LoraConfig(r=64, ...)
# Puis utilisez le meme Trainer qu'en section 5

resultats_comparaison = {
    "r=4": {"parametres": None, "perte": None},   # TODO etudiant : remplir
    "r=16": {"parametres": None, "perte": None},   # reference (deja fait)
    "r=64": {"parametres": None, "perte": None},   # TODO etudiant : remplir
}
print("Completez les entrainements puis remplissez resultats_comparaison")

Completez les entrainements puis remplissez resultats_comparaison


### Exercice 3 : Analyser les limites du SFT avec un petit dataset

Produisez un petit rapport (en markdown dans une cellule) analysant les limites observees :
1. Quels types d'instructions le modele SFT gere-t-il bien ? Mal ?
2. Que se passe-t-il si vous donnez une instruction hors du dataset d'entrainement ?
3. Combien d'exemples faudrait-il approximativement pour obtenir des resultats fiables ?

**Indice** : Testez avec des instructions originales non presentes dans le dataset pour evaluer la generalisation.

In [14]:
# Exercice 3 : Test de generalisation
# TODO etudiant : testez le modele SFT avec des instructions originales

instructions_test = [
    "Quelle est la capitale de l'Australie ?",
    "Expliquez pourquoi le ciel est bleu.",
    "Donnez 2 astuces pour apprendre une langue etrangere.",
]

# TODO etudiant : chargez le meilleur modele SFT, testez ces instructions
# et analysez la qualite des reponses
print("Testez avec votre modele SFT et analysez les resultats")

Testez avec votre modele SFT et analysez les resultats


## 9. Resume du FT-03

| Concept | Detail |
|---------|--------|
| **SFT (Supervised Fine-Tuning)** | Entrainement sur des paires instruction/reponse pour transformer un modele de base en assistant |
| **Chat template** | Format de conversation (`### Human:` / `### Assistant:`) — contrat entre entrainement et inference |
| **Dataset SFT** | 21 paires en francais couvrant 6 categories de taches |
| **QLoRA** | Quantization 4-bit NF4 + LoRA (r=16) = ~0.1% des parametres entraines |
| **Resultat** | Le modele apprend a repondre aux instructions au lieu de continuer le texte |
| **Limites** | Petit dataset = generalisation limitee, reponses parfois repetitives |

**Prochaines etapes** : Le FT-04 explorera le RLHF (Reinforcement Learning from Human Feedback), qui affine les reponses du modele SFT en utilisant les preferences humaines comme signal de recompense.